In [1]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [2]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

In [3]:
orders["delivery_days"].isnull().sum()

np.int64(2965)

In [4]:
orders_clean = orders.dropna(subset=["delivery_days"])
orders_clean.shape

(96476, 9)

2,965 orders (~3%) have no delivery_days value because order_delivered_customer_date 
is missing — these orders were never delivered (likely cancelled or lost). Since 
there's no real underlying delivery time to estimate for an order that was never 
delivered, imputing a value would fabricate data. Decision: remove these rows from 
the modeling dataset via dropna().

In [5]:
orders_clean["is_outlier_delivery"] = orders_clean["delivery_days"] > 60
orders_clean["is_outlier_delivery"].sum()

np.int64(288)

In [6]:
orders_clean.duplicated().sum()
items.duplicated().sum()
customers.duplicated().sum()
sellers.duplicated().sum()
products.duplicated().sum()

np.int64(0)

In [7]:
orders_clean["order_approved_at"] = pd.to_datetime(orders_clean["order_approved_at"])
orders_clean["order_delivered_carrier_date"] = pd.to_datetime(orders_clean["order_delivered_carrier_date"])
orders_clean["order_estimated_delivery_date"] = pd.to_datetime(orders_clean["order_estimated_delivery_date"])

In [8]:
orders_clean.info()

<class 'pandas.DataFrame'>
Index: 96476 entries, 0 to 99440
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96476 non-null  str           
 1   customer_id                    96476 non-null  str           
 2   order_status                   96476 non-null  str           
 3   order_purchase_timestamp       96476 non-null  datetime64[us]
 4   order_approved_at              96462 non-null  datetime64[us]
 5   order_delivered_carrier_date   96475 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  96476 non-null  datetime64[us]
 8   delivery_days                  96476 non-null  float64       
 9   is_outlier_delivery            96476 non-null  bool          
dtypes: bool(1), datetime64[us](5), float64(1), str(3)
memory usage: 7.5 MB


In [9]:
orders_clean["is_late"] = orders_clean["order_delivered_customer_date"] > orders_clean["order_estimated_delivery_date"]
orders_clean["is_late"].sum()

np.int64(7827)

In [10]:
orders_clean["purchase_month"] = orders_clean["order_purchase_timestamp"].dt.month
orders_clean["purchase_dayofweek"] = orders_clean["order_purchase_timestamp"].dt.dayofweek

In [11]:
orders_clean[["purchase_month", "purchase_dayofweek"]].head()

,purchase_month,purchase_dayofweek
0,10,0
1,7,1
2,8,2
3,11,5
4,2,1


In [12]:
orders_clean.groupby("purchase_month")["is_late"].mean()

purchase_month
1     0.062284
2     0.134243
3     0.171536
4     0.059554
5     0.066446
6     0.022099
7     0.040786
8     0.075778
9     0.052277
10    0.050548
11    0.143112
12    0.083787
Name: is_late, dtype: float64

In [13]:
orders_clean["purchase_month"].value_counts().sort_index()

purchase_month
1      7819
2      8209
3      9549
4      9101
5     10294
6      9231
7     10028
8     10544
9      4151
10     4748
11     7288
12     5514
Name: count, dtype: int64

## Seasonal Late-Delivery Pattern

Late delivery rate varies significantly by month, ranging from 2.2% (June) to 17.2% 
(March). November also shows an elevated rate (14.3%), plausibly linked to Black 
Friday/Cyber Monday order surges. February and March's high rates may relate to 
Brazilian Carnival season, though this requires further verification against exact 
holiday dates. June's low rate is backed by a normal sample size (9,231 orders), 
confirming it's a genuine pattern, not statistical noise from a small sample.

In [14]:
writer = pd.ExcelWriter("../reports/cleaning_report.xlsx", engine="openpyxl")

In [15]:
%whos DataFrame

Variable       Type         Data/Info
-------------------------------------
customers      DataFrame    Shape: (99441, 5)
items          DataFrame    Shape: (112650, 7)
orders         DataFrame    Shape: (99441, 9)
orders_clean   DataFrame    Shape: (96476, 13)
products       DataFrame    Shape: (32951, 9)
sellers        DataFrame    Shape: (3095, 4)


In [16]:
%whos

Variable       Type              Data/Info
------------------------------------------
customers      DataFrame         Shape: (99441, 5)
items          DataFrame         Shape: (112650, 7)
orders         DataFrame         Shape: (99441, 9)
orders_clean   DataFrame         Shape: (96476, 13)
pd             module            <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>
plt            module            <module 'matplotlib.pyplo<...>\\matplotlib\\pyplot.py'>
products       DataFrame         Shape: (32951, 9)
sellers        DataFrame         Shape: (3095, 4)
writer         OpenpyxlWriter    <pandas.io.excel._openpyx<...>ct at 0x00000184C64CD820>


## Final Data Cleaning Summary & Validation

To ensure the dataset is fully prepared for machine learning models (Linear Regression, Random Forest, and XGBoost), the following validation and refinement steps were performed:

1. **Target Variable Cleanup:**
   - Removed 2,965 orders (~3%) missing `order_delivered_customer_date` because undelivered/cancelled orders do not have a true delivery duration[cite: 1].
   - Filtered out impossible negative delivery days (`delivery_days < 0`) caused by upstream logging anomalies.

2. **Outlier Management:**
   - Identified and capped/filtered delivery durations exceeding 60 days to prevent extreme postal loss delays from distorting regression models (RMSE/MAE).

3. **Data Type Standardization:**
   - Converted all raw timestamp strings (`order_purchase_timestamp`, `order_delivered_customer_date`, etc.) to standard `datetime64` format[cite: 1].

4. **Category & Metadata Imputation:**
   - Merged product category English translations and imputed missing category entries with `'unknown'`.

5. **Exporting Processed Dataset:**
   - Saved the cleaned baseline dataset to `../data/processed/processed_data.csv` for seamless ingestion into downstream EDA and model development pipelines.

In [18]:
# Final Data Cleansing & Validation Steps

# 1. Drop missing target values & impossible negative delivery days
orders_clean = orders.dropna(subset=["delivery_days"]).copy()
orders_clean = orders_clean[orders_clean["delivery_days"] >= 0]

# 2. Filter extreme delivery outliers (> 60 days) to prevent model distortion
orders_clean = orders_clean[orders_clean["delivery_days"] <= 60]

# 3. Clean product category translation missing values
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
products_clean = products.merge(category_translation, on='product_category_name', how='left')
products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna('unknown')

# 4. Save processed base dataset
orders_clean.to_csv('../data/processed/processed_data.csv', index=False)
print("Data cleaning finalized successfully! Shape:", orders_clean.shape)

Data cleaning finalized successfully! Shape: (96188, 9)


 ## Group Multi-Item Orders (Prevent Data Duplication)
 
Some orders in the Olist dataset contain multiple items (e.g., 2 shirts in one order), which creates duplicate order_id rows in olist_order_items_dataset.csv. Grouping them ensures every order remains exactly one row when merged:

In [19]:
# Aggregate order items to keep exactly 1 row per order_id
items_aggregated = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()

print("Aggregated order items shape:", items_aggregated.shape)

Aggregated order items shape: (98666, 4)


## Clean & Deduplicate Geolocation Data
The geolocation dataset contains millions of rows with slight coordinate variations for the same zip code prefix. Averaging coordinates by zip code prevents duplicate matches when merging customer/seller locations:

In [20]:
# Load geolocation data
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

# Average lat/lng per zip code prefix to remove duplicates
geo_clean = geolocation.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "mean"),
    lng=("geolocation_lng", "mean")
).reset_index()

print("Cleaned unique zip code coordinates:", geo_clean.shape)

Cleaned unique zip code coordinates: (19015, 3)


## Final Merged Model Dataset Export
Merge all cleaned subsets into a single, clean master file saved to ../data/processed/processed_data.csv:

In [21]:
# Merge clean orders with aggregated items and customer/seller info
master_clean = orders_clean.merge(items_aggregated, on="order_id", how="left") \
                           .merge(customers[["customer_id", "customer_state", "customer_zip_code_prefix"]], on="customer_id", how="left")

# Save master dataset for modeling
master_clean.to_csv("../data/processed/processed_data.csv", index=False)
print("Master cleaned dataset saved! Final Shape:", master_clean.shape)

Master cleaned dataset saved! Final Shape: (96188, 14)


In [22]:
%whos DataFrame

Variable               Type         Data/Info
---------------------------------------------
category_translation   DataFrame    Shape: (71, 2)
customers              DataFrame    Shape: (99441, 5)
geo_clean              DataFrame    Shape: (19015, 3)
geolocation            DataFrame    Shape: (1000163, 5)
items                  DataFrame    Shape: (112650, 7)
items_aggregated       DataFrame    Shape: (98666, 4)
master_clean           DataFrame    Shape: (96188, 14)
orders                 DataFrame    Shape: (99441, 9)
orders_clean           DataFrame    Shape: (96188, 9)
products               DataFrame    Shape: (32951, 9)
products_clean         DataFrame    Shape: (32951, 10)
sellers                DataFrame    Shape: (3095, 4)


In [23]:
master_clean.to_csv("../data/processed/processed_data.csv", index=False)
print("Master cleaned dataset saved successfully!")

Master cleaned dataset saved successfully!
